# 19회. 파일과 폴더 — 측정 파일 여러 개를 코드로

**오늘의 목표**
1. `pathlib`로 **경로**를 만들고, 폴더 안의 **파일 목록**을 얻는다.
2. 텍스트 파일을 **읽고 쓴다** (인코딩 `utf-8`).
3. `csv` 모듈로 CSV 파일을 읽고, **파일마다 요약**한다.
4. 요약 결과를 **엑셀에서 열리는 CSV**로 저장한다.
5. 파일을 **규칙에 맞는 이름으로 복사**한다.

> 📁 이 노트북은 같은 폴더의 `data/` 폴더를 사용합니다. 노트북과 `data` 폴더를 **함께** `ai-black/01_practice/week10/`으로 복사하세요.
> `data/`의 측정 파일은 모두 **가상 데이터**입니다.

> ✅ **해답 노트북** — 연습 문제의 예시 답안이 채워져 있습니다.

## 1. 경로 다루기 — pathlib
07회에 배운 경로를 코드로 다룹니다. `Path`를 쓰면 Mac(`/`)과 Windows(`\\`)의 차이를 신경 쓰지 않아도 됩니다.

In [ ]:
from pathlib import Path

here = Path.cwd()                  # 지금 노트북이 실행되는 폴더
data_dir = here / "data"           # 경로 이어 붙이기는 / 연산자
print("지금 위치:", here)
print("데이터 폴더:", data_dir)
print("있나?", data_dir.exists())

## 2. 폴더 안 파일 목록
| 코드 | 결과 |
|---|---|
| `data_dir.iterdir()` | 폴더 안의 모든 항목 |
| `data_dir.glob("*.csv")` | 이름이 `.csv`로 끝나는 것만 (`*`는 "아무 글자") |
| `p.name` / `p.stem` / `p.suffix` | `0408_1층교실.csv` / `0408_1층교실` / `.csv` |

In [ ]:
for p in sorted(data_dir.iterdir()):
    print(p.name)

print("---- CSV만 ----")
csv_files = sorted(data_dir.glob("*.csv"))
for p in csv_files:
    print(p.stem, "|", p.suffix)
print("CSV 파일 수:", len(csv_files))

## 3. 텍스트 파일 읽고 쓰기
- **인코딩(encoding)**: 글자를 파일에 저장하는 규칙. 한글이 깨지지 않게 항상 `encoding="utf-8"`을 씁니다.
- `with open(...) as f:` 블록이 끝나면 파일이 **자동으로 닫힙니다.**

In [ ]:
memo_path = data_dir / "측정메모.txt"
with open(memo_path, encoding="utf-8") as f:
    text = f.read()
print(text)

In [ ]:
# 줄 단위로 읽어서, '-'로 시작하는 줄(측정 지점 메모)만 고르기
with open(memo_path, encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line.startswith("-"):
            print(line[2:])

### 🧪 연습 1
`측정메모.txt`에서 **"인원"** 또는 **"원아"** 라는 글자가 들어간 줄만 골라 출력하세요.

In [ ]:
with open(memo_path, encoding="utf-8") as f:
    for line in f:
        if "인원" in line or "원아" in line:
            print(line.strip())

### 파일 쓰기
- `"w"`: 새로 쓰기 (**같은 이름 파일이 있으면 덮어씀!**)
- `"a"`: 이어 쓰기

In [ ]:
out_dir = here / "output"
out_dir.mkdir(exist_ok=True)       # 없으면 만들고, 있으면 그냥 둠

log_path = out_dir / "작업기록.txt"
with open(log_path, "w", encoding="utf-8") as f:
    f.write("19회 실습 작업 기록\n")
    f.write(f"CSV 파일 {len(csv_files)}개 확인\n")

print(log_path.read_text(encoding="utf-8"))

## 4. CSV 파일 읽기
CSV는 **쉼표로 구분된 표**입니다. 메모장으로 열어 보면 이렇게 생겼습니다.
```
DateTime,CO2_ppm,Temp_C,RH
2025-04-08 10:02,723,21.4,33.8
...
```
`csv.DictReader`는 각 줄을 **{열 이름: 값} 딕셔너리**로 줍니다. (14회 딕셔너리의 리스트!)
⚠️ CSV에서 읽은 값은 모두 **문자열**입니다 → 계산하려면 `int()`, `float()` (13회)

In [ ]:
import csv

first = csv_files[0]
with open(first, encoding="utf-8") as f:
    reader = csv.DictReader(f)
    rows = list(reader)

print("파일:", first.name, "/ 줄 수:", len(rows))
print("첫 줄:", rows[0])
print("CO2 값의 자료형:", type(rows[0]["CO2_ppm"]))

### 🧪 연습 2
위에서 읽은 `rows`에서 CO2 값만 **정수 리스트**로 만들고, 평균(소수 1자리)과 최댓값을 출력하세요.

In [ ]:
co2 = [int(r["CO2_ppm"]) for r in rows]
print("평균:", round(sum(co2) / len(co2), 1))
print("최대:", max(co2))

## 5. 모든 파일 요약하기 — 폴더 → 요약표
16회 반복문 + 17회 함수 + 오늘 배운 파일 읽기를 **조립**합니다.

In [ ]:
def read_co2(path):
    """측정 CSV 파일에서 CO2 값을 정수 리스트로 읽는다."""
    with open(path, encoding="utf-8") as f:
        return [int(r["CO2_ppm"]) for r in csv.DictReader(f)]

def summarize(values, limit=1000):
    mean = round(sum(values) / len(values), 1)
    return {"count": len(values), "mean": mean, "max": max(values),
            "min": min(values), "result": "부적합" if mean >= limit else "적합"}

summary = []
for path in csv_files:
    date, place = path.stem.split("_")         # '0408_1층교실' → '0408', '1층교실'
    s = summarize(read_co2(path))
    summary.append({"file": path.name, "date": date, "place": place, **s})

for row in summary:
    print(row)

> 💡 `{"file": ..., **s}` 의 `**s`는 딕셔너리 `s`의 내용을 **풀어서 합치는** 문법입니다. AI 코드에 자주 보입니다.

## 6. 요약표를 CSV로 저장 — 엑셀에서 한글이 깨지지 않게
엑셀은 `utf-8` CSV의 한글을 깨뜨리는 경우가 있습니다. **`utf-8-sig`** 로 저장하면 엑셀에서도 바로 열립니다.

In [ ]:
summary_path = out_dir / "CO2_요약.csv"
with open(summary_path, "w", newline="", encoding="utf-8-sig") as f:
    writer = csv.DictWriter(f, fieldnames=summary[0].keys())
    writer.writeheader()
    writer.writerows(summary)

print("저장:", summary_path)
print(summary_path.read_text(encoding="utf-8-sig"))

### 🧪 연습 3
`summary`에서 **부적합인 지점만** 골라 `output/CO2_부적합.csv`로 저장하세요. 저장한 파일 내용을 출력해 확인하세요.

In [ ]:
failed = [row for row in summary if row["result"] == "부적합"]
failed_path = out_dir / "CO2_부적합.csv"
with open(failed_path, "w", newline="", encoding="utf-8-sig") as f:
    writer = csv.DictWriter(f, fieldnames=failed[0].keys())
    writer.writeheader()
    writer.writerows(failed)
print(failed_path.read_text(encoding="utf-8-sig"))

## 7. 파일 이름 규칙에 맞게 복사하기
측정기에서 받은 파일 이름(`0408_1층교실.csv`)을 보고서 규칙(`2025-04-08_1층교실_CO2.csv`)으로 바꿔 **복사**합니다.
원본은 그대로 두고 **복사본**을 만드는 것이 안전합니다. (`shutil.copy`)

In [ ]:
import shutil

renamed_dir = out_dir / "renamed"
renamed_dir.mkdir(exist_ok=True)

for path in csv_files:
    date, place = path.stem.split("_")
    new_name = f"2025-{date[:2]}-{date[2:]}_{place}_CO2.csv"
    shutil.copy(path, renamed_dir / new_name)
    print(path.name, "→", new_name)

### 🧪 연습 4 — AI와 함께
AI에게 이렇게 물어보세요:
> "파이썬 pathlib로 폴더 안의 모든 .csv 파일 크기를 KB 단위로 출력하는 코드를 초보자용 주석과 함께 알려 줘."

받은 코드를 `data_dir`에 맞게 고쳐 실행하세요.

In [ ]:
for path in sorted(data_dir.glob("*.csv")):
    size_kb = path.stat().st_size / 1024        # 파일 크기(바이트) → KB
    print(f"{path.name:<20} {size_kb:.1f} KB")

---
## 정리
| 하고 싶은 일 | 코드 |
|---|---|
| 경로 만들기 | `Path.cwd() / "data"` |
| 파일 목록 | `sorted(data_dir.glob("*.csv"))` |
| 이름 조각 | `p.name`, `p.stem`, `p.suffix` |
| 텍스트 읽기·쓰기 | `open(p, encoding="utf-8")`, `"w"` / `"a"` |
| CSV 읽기 | `csv.DictReader(f)` → 딕셔너리들 (값은 **문자열**) |
| 엑셀용 CSV 저장 | `encoding="utf-8-sig"`, `csv.DictWriter` |
| 폴더 만들기 / 복사 | `mkdir(exist_ok=True)`, `shutil.copy(원본, 사본)` |

> 4단계에서는 오늘 한 일을 **pandas** 한두 줄로 합니다 (`pd.read_csv`, `to_csv`). 오늘 배운 것은 그 안에서 실제로 일어나는 일입니다.

**다음 시간:** 20회 오류 해결하기 — 오류 메시지를 읽고 AI와 함께 고치기